In [1]:
import osmnx as ox
import pandas as pd

In [2]:

# Target area
place_name = "Nancy, France"

# Tags to extract for the LLM dataset
tags = {
    'amenity': True,         # Restaurants, cafes, schools, etc.
    'historic': True,        # Monuments and heritage sites
    'tourism': ['museum', 'hotel', 'attraction'],
    'leisure': 'park'        # Green spaces
}

print(f"Retrieving geospatial features for {place_name}...")

Retrieving geospatial features for Nancy, France...


In [3]:

pois = ox.features_from_place(place_name, tags)

pois_points = pois[pois.geometry.type == 'Point'].copy()

In [4]:

pois_points['latitude'] = pois_points.geometry.y
pois_points['longitude'] = pois_points.geometry.x

In [5]:
# Gather necessary or possibly useful information
data_columns = ['name', 'amenity','latitude', 'longitude']
df_nancy = pois_points[[col for col in data_columns if col in pois_points.columns]]

In [6]:

# Preview the first few rows
df_nancy.head(10)


df_nancy.info()

<class 'pandas.DataFrame'>
MultiIndex: 5468 entries, ('node', np.int64(262840256)) to ('node', np.int64(13764400350))
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1200 non-null   str    
 1   amenity    5324 non-null   str    
 2   latitude   5468 non-null   float64
 3   longitude  5468 non-null   float64
dtypes: float64(2), str(2)
memory usage: 259.1+ KB


In [7]:
df_nancy = df_nancy.dropna()

In [8]:
df_nancy.info()

<class 'pandas.DataFrame'>
MultiIndex: 1112 entries, ('node', np.int64(313719091)) to ('node', np.int64(13728398028))
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype  
---  ------     --------------  -----  
 0   name       1112 non-null   str    
 1   amenity    1112 non-null   str    
 2   latitude   1112 non-null   float64
 3   longitude  1112 non-null   float64
dtypes: float64(2), str(2)
memory usage: 110.2+ KB


Sentence Generation process:
iterate through the dataframe.
for every other location in the dataframe generate a direction
Create result sentences in format: Name2 is DIRECTION of Name1

In [9]:
df_nancy['name'].iloc[0]

'Place Commanderie'

In [11]:
sentence_list = []
for i in range(len(df_nancy)):
    for j in range(len(df_nancy)):
        if df_nancy['name'].iloc[i] != df_nancy['name'].iloc[j]:
            name1 = df_nancy['name'].iloc[i]
            name2 = df_nancy['name'].iloc[j]
            lat1 = df_nancy['latitude'].iloc[i]
            lat2 = df_nancy['latitude'].iloc[j]
            lon1 = df_nancy['longitude'].iloc[i]
            lon2 = df_nancy['longitude'].iloc[j]
            ammen1 = df_nancy['amenity'].iloc[i]
            ammen2 = df_nancy['amenity'].iloc[j]
            lat = df_nancy['latitude'].iloc[j] - df_nancy['latitude'].iloc[i]
            lon = df_nancy['longitude'].iloc[j] - df_nancy['longitude'].iloc[i]
            cardinal = ""
            if abs(lat) >= abs(lon):
                if lat > 0:
                    cardinal = 'NORTH'
                else:
                    cardinal = 'SOUTH'
            else:
                if lon > 0:
                    cardinal = 'EAST'
                else:
                    cardinal = 'WEST'
            sentence = f'{df_nancy['name'].iloc[j]} is {cardinal} of {df_nancy['name'].iloc[i]}'
            sentence_list.append({'sentence':sentence , 'cardinal':cardinal ,'name1':name1 , 'name2':name2 ,
            'lat1':lat1 , 'lon1':lon1, 'lat2': lat2, 'lon2':lon2, 'amenity1':ammen1 ,'amenity2':ammen2})

In [12]:
len(sentence_list)

1234828

In [13]:
sentence_list[100:106]

[{'sentence': 'Les Oliviers is EAST of Place Commanderie',
  'cardinal': 'EAST',
  'name1': 'Place Commanderie',
  'name2': 'Les Oliviers',
  'lat1': np.float64(48.6858518),
  'lon1': np.float64(6.1674456),
  'lat2': np.float64(48.695614),
  'lon2': np.float64(6.177814),
  'amenity1': 'bicycle_rental',
  'amenity2': 'restaurant'},
 {'sentence': 'Akoya Sushi is NORTH of Place Commanderie',
  'cardinal': 'NORTH',
  'name1': 'Place Commanderie',
  'name2': 'Akoya Sushi',
  'lat1': np.float64(48.6858518),
  'lon1': np.float64(6.1674456),
  'lat2': np.float64(48.7011832),
  'lon2': np.float64(6.1765845),
  'amenity1': 'bicycle_rental',
  'amenity2': 'restaurant'},
 {'sentence': 'Pharmacie de la Croix is NORTH of Place Commanderie',
  'cardinal': 'NORTH',
  'name1': 'Place Commanderie',
  'name2': 'Pharmacie de la Croix',
  'lat1': np.float64(48.6858518),
  'lon1': np.float64(6.1674456),
  'lat2': np.float64(48.7025757),
  'lon2': np.float64(6.1756487),
  'amenity1': 'bicycle_rental',
  'ame

In [14]:
sentence_df = pd.DataFrame(sentence_list)
sentence_df


,sentence,cardinal,name1,name2,lat1,lon1,lat2,lon2,amenity1,amenity2
0,Commanderie - Chalnot is EAST of Place Command...,EAST,Place Commanderie,Commanderie - Chalnot,48.685852,6.167446,48.686859,6.172236,bicycle_rental,bicycle_rental
1,Gare Saint-Léon is EAST of Place Commanderie,EAST,Place Commanderie,Gare Saint-Léon,48.685852,6.167446,48.689593,6.172298,bicycle_rental,bicycle_rental
2,Saint-Sébastien - Saint-Thiébaut is EAST of Pl...,EAST,Place Commanderie,Saint-Sébastien - Saint-Thiébaut,48.685852,6.167446,48.688691,6.179350,bicycle_rental,bicycle_rental
3,Marché Central is EAST of Place Commanderie,EAST,Place Commanderie,Marché Central,48.685852,6.167446,48.689170,6.183843,bicycle_rental,bicycle_rental
4,Place Dombasle is EAST of Place Commanderie,EAST,Place Commanderie,Place Dombasle,48.685852,6.167446,48.691851,6.178190,bicycle_rental,bicycle_rental
...,...,...,...,...,...,...,...,...,...,...
1234823,AMOS ESDAC is NORTH of GeoRessources,NORTH,GeoRessources,AMOS ESDAC,48.673216,6.171415,48.689768,6.180073,research_institute,college
1234824,WIN Sport School Nancy is NORTH of GeoRessources,NORTH,GeoRessources,WIN Sport School Nancy,48.673216,6.171415,48.697291,6.173682,research_institute,college
1234825,Dr Jean-Pascal FYAD is NORTH of GeoRessources,NORTH,GeoRessources,Dr Jean-Pascal FYAD,48.673216,6.171415,48.694924,6.181452,research_institute,doctors
1234826,La Petite Farandole is EAST of GeoRessources,EAST,GeoRessources,La Petite Farandole,48.673216,6.171415,48.690948,6.197476,research_institute,kindergarten


In [15]:
sentence_df.to_csv('directionSentences.csv', sep = ';')